In [ ]:
# * | // MARK: 1. 환경 설정 및 MySQL 서버 시작 =====================================
print("MySQL 서버를 설치 및 시작합니다...")
!apt-get update
!apt-get install mysql-server
!service mysql start

# SQL 실행을 간소화하기 위한 함수 (표 형식 출력 -t)
def sql(query):
    !mysql -t -e "{query}"

# * | // MARK: 2. 데이터베이스 및 테이블 설계 (스키마 생성) =====================================
# 기존 DB가 있다면 삭제 후 재생성
sql("DROP DATABASE IF EXISTS mart_db;")
sql("CREATE DATABASE mart_db;")

# 유저 테이블
sql("""
CREATE TABLE mart_db.users (
    u_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(20),
    city VARCHAR(20),
    level VARCHAR(10) DEFAULT 'Silver'
);
""")

# 상품 테이블
sql("""
CREATE TABLE mart_db.products (
    p_id INT AUTO_INCREMENT PRIMARY KEY,
    p_name VARCHAR(50),
    category VARCHAR(20),
    price INT,
    stock INT
);
""")

# 주문 내역 테이블 (유저와 상품 연결)
sql("""
CREATE TABLE mart_db.orders (
    o_id INT AUTO_INCREMENT PRIMARY KEY,
    u_id INT,
    p_id INT,
    amount INT,
    o_date DATE,
    FOREIGN KEY (u_id) REFERENCES mart_db.users(u_id),
    FOREIGN KEY (p_id) REFERENCES mart_db.products(p_id)
);
""")

# * | // MARK: 3. 다채로운 데이터 삽입 (Bulk Insert) =====================================
print("데이터를 삽입 중입니다...")

sql("""
INSERT INTO mart_db.users (name, city, level) VALUES 
('이몽룡', '서울', 'Gold'), ('성춘향', '남원', 'VIP'), 
('홍길동', '한양', 'Silver'), ('임꺽정', '구월산', 'Gold'),
('심청', '백령도', 'Silver');
""")

sql("""
INSERT INTO mart_db.products (p_name, category, price, stock) VALUES 
('갤럭시 S24', '가전', 1200000, 15), ('아이폰 15', '가전', 1300000, 10),
('불닭볶음면', '식품', 1500, 100), ('삼다수', '식품', 800, 200),
('기계식 키보드', '잡화', 150000, 5), ('캠핑 의자', '잡화', 45000, 0);
""")

sql("""
INSERT INTO mart_db.orders (u_id, p_id, amount, o_date) VALUES 
(1, 1, 1, '2024-05-10'), (2, 3, 5, '2024-05-11'),
(3, 4, 10, '2024-05-12'), (1, 5, 1, '2024-05-13'),
(4, 2, 1, '2024-05-14'), (2, 1, 1, '2024-05-15');
""")

# * | // MARK: 4. 실무형 데이터 분석 쿼리 (JOIN & Aggregation) =====================================

print("\n[리포트 1] 전체 주문 상세 내역 (JOIN)")
sql("""
SELECT 
    o.o_id AS '번호', 
    u.name AS '고객명', 
    p.p_name AS '상품명', 
    p.price AS '단가', 
    o.amount AS '수량',
    (p.price * o.amount) AS '총결제액'
FROM mart_db.orders o
JOIN mart_db.users u ON o.u_id = u.u_id
JOIN mart_db.products p ON o.p_id = p.p_id;
""")

print("\n[리포트 2] 카테고리별 매출 합계 및 평균 단가")
sql("""
SELECT 
    category AS '카테고리', 
    SUM(price * stock) AS '재고총액',
    COUNT(*) AS '상품수',
    ROUND(AVG(price), 0) AS '평균가격'
FROM mart_db.products
GROUP BY category;
""")

print("\n[리포트 3] VIP 고객의 주문 현황")
sql("""
SELECT u.name, u.level, p.p_name, o.o_date
FROM mart_db.orders o
JOIN mart_db.users u ON o.u_id = u.u_id
JOIN mart_db.products p ON o.p_id = p.p_id
WHERE u.level = 'VIP';
""")

print("\n[리포트 4] 현재 품절이거나 재고가 10개 미만인 위험 상품")
sql("SELECT p_name, stock FROM mart_db.products WHERE stock < 10;")

# * | // MARK: 5. 데이터 수정 =====================================
print("\n[업데이트] '삼다수' 가격 인상 및 '성춘향' 등급 조정")
sql("UPDATE mart_db.products SET price = 1000 WHERE p_name = '삼다수';")
sql("UPDATE mart_db.users SET level = 'VVIP' WHERE name = '성춘향';")

print("\n[최종 결과 확인]")
sql("SELECT * FROM mart_db.products WHERE p_name = '삼다수';")
sql("SELECT * FROM mart_db.users WHERE name = '성춘향';")

# * | // MARK: 6. 데이터 삭제 및 정리 (Delete) =====================================
print("\n[삭제 시나리오 1] 재고가 0인 상품을 목록에서 제거합니다.")
# '캠핑 의자'가 재고가 0이므로 삭제 대상이 됩니다.
sql("DELETE FROM mart_db.products WHERE stock = 0;")

print("\n[삭제 시나리오 2] '심청' 고객이 개인정보 보호를 위해 탈퇴를 요청했습니다.")
# 주의: 외래 키 제약 조건 때문에 주문 내역이 있는 유저는 바로 삭제가 안 될 수 있습니다.
# 여기서는 주문 내역이 없는 '심청' 데이터를 안전하게 삭제합니다.
sql("DELETE FROM mart_db.users WHERE name = '심청';")

print("\n[최종 확인] 남은 상품 리스트")
sql("SELECT p_name, stock, price FROM mart_db.products;")

print("\n[최종 확인] 남은 유저 리스트")
sql("SELECT name, level FROM mart_db.users;")


# * | // MARK: 7. 고급 분석: 등급별 타겟팅 리포트 =====================================
print("\n[리포트 5] 고객 등급별 구매 권장 문구 (CASE 문)")
sql("""
SELECT 
    name, 
    level, 
    CASE 
        WHEN level = 'VVIP' THEN '전담 매니저 배정 대상'
        WHEN level = 'Gold' THEN '10% 할인 쿠폰 발급'
        ELSE '신규 가입 이벤트 안내'
    END AS '마케팅_액션'
FROM mart_db.users;
""")

# * | // MARK: 8. 데이터 무결성 테스트 (에러 케이스) =====================================
print("\n[테스트] 존재하지 않는 상품(999번)을 주문 시도 (외래키 제약조건 작동 확인)")
# 이 쿼리는 에러가 발생해야 정상입니다. (데이터 무결성 증명)
try:
    sql("INSERT INTO mart_db.orders (u_id, p_id, amount, o_date) VALUES (1, 999, 1, '2024-05-20');")
except:
    print(">> 성공: 외래 키 제약 조건에 의해 잘못된 주문이 차단되었습니다.")